# Step 7 — Visualization & GeoJSON Export

Reconstructs one flight using all three methods and exports to geojson.io.

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6

## Cell 1 — Setup: load model and flight list

In [1]:
import sys, os, json
from pathlib import Path
import torch
import pandas as pd
import numpy as np
from pyproj import Geod
from scipy.interpolate import interp1d

# ── Must chdir to noel_part BEFORE importing reconstruction ───────────────────
NOEL_DIR = Path("../noel_part").resolve()
os.chdir(NOEL_DIR)
sys.path.insert(0, str(NOEL_DIR))
sys.path.append(str(NOEL_DIR.parent / "notebook"))

print(f"Working dir: {os.getcwd()}")

BASE_DIR  = Path(".")
CLEAN_DIR = BASE_DIR / "cleaned_data_final"
RECON_DIR = BASE_DIR / "final_reconstructions"
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

geod = Geod(ellps="WGS84")

# ─────────────────────────────────────────────────────────────
# LOAD GRU MODEL (same as step 5 evaluation)
# ─────────────────────────────────────────────────────────────
from step5_train_gru_v2 import TrajectoryGRU, gc_interpolate_batch
from step3_baseline import reconstruct_gap_baseline

MODEL_PATH = Path(r"C:\Users\marko\Desktop\AeroML3(og)\AeroML3\step5_v2\best_model_v2.pt")
NORM_PATH  = Path(r"C:\Users\marko\Desktop\AeroML3(og)\AeroML3\artifacts\step4_ml_dataset\dataset\normalization_stats.json")

with open(NORM_PATH) as f:
    NORM = json.load(f)

BEFORE_STEPS = NORM["before_steps"]   # 64
AFTER_STEPS  = NORM["after_steps"]    # 32
RESAMPLE_SEC = NORM["resample_sec"]   # 60

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_gru = TrajectoryGRU().to(device)
model_gru.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model_gru.eval()

flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]

print(f"Model loaded on {device}")
print(f"Flights available: {len(flights)}")
print(f"GRU: BEFORE_STEPS={BEFORE_STEPS}, AFTER_STEPS={AFTER_STEPS}, RESAMPLE_SEC={RESAMPLE_SEC}s")

c:\Users\marko\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Working dir: C:\Users\marko\Desktop\AeroML3(og)\AeroML3\noel_part
Model loaded on cpu
Flights available: 2149
GRU: BEFORE_STEPS=64, AFTER_STEPS=32, RESAMPLE_SEC=60s


## Cell 2 — Select a specific flight

We use a known Ireland→New York transatlantic flight for a clean visualization.

In [42]:
# ── Select a flight to visualize ──────────────────────────────────────────────
# Run Cell 6 first to see a ranked list of flights.
# Paste any FLIGHT_NAME from that table here.
# Flights with high improvement_mean_pct show the clearest Kalman vs baseline gap.

FLIGHT_NAME = "20240312_346114_174632_193401"

# Find which step folder contains this flight
STEP_NAME = None
for s in CLEAN_DIR.iterdir():
    if s.is_dir() and (s / FLIGHT_NAME).exists():
        STEP_NAME = s.name
        break

if STEP_NAME is None:
    raise FileNotFoundError(
        f"Flight '{FLIGHT_NAME}' not found under {CLEAN_DIR}\n"
        "Run Cell 6 to get valid flight names."
    )

FLIGHT_DIR = CLEAN_DIR / STEP_NAME / FLIGHT_NAME
print(f"Flight : {FLIGHT_NAME}")
print(f"Step   : {STEP_NAME}")

# ── Load data ─────────────────────────────────────────────────────────────────
def _find(fd, names):
    return next((fd / n for n in names if (fd / n).exists()), None)

_ap  = _find(FLIGHT_DIR, ["adsc_clean.parquet",        "adsc.parquet"])
_bp  = _find(FLIGHT_DIR, ["adsb_before_clean.parquet", "adsb_before.parquet"])
_afp = _find(FLIGHT_DIR, ["adsb_after_clean.parquet",  "adsb_after.parquet"])

adsc = pd.read_parquet(str(_ap)).dropna(subset=["latitude","longitude"])
bef  = pd.read_parquet(str(_bp)).dropna(subset=["latitude","longitude"])
aft  = pd.read_parquet(str(_afp)).dropna(subset=["latitude","longitude"])

for _df in [adsc, bef, aft]:
    _df["timestamp"] = pd.to_datetime(_df["timestamp"], utc=True).dt.tz_localize(None)
    _df.sort_values("timestamp", inplace=True)
    _df.reset_index(drop=True, inplace=True)

# ── Gap boundaries ────────────────────────────────────────────────────────────
t_gap_start = bef["timestamp"].iloc[-1]
t_gap_end   = aft["timestamp"].iloc[0]
gap_min     = (t_gap_end - t_gap_start).total_seconds() / 60

bef_trim = bef[bef["timestamp"] >= t_gap_start - pd.Timedelta(minutes=30)].reset_index(drop=True)
aft_trim = aft[aft["timestamp"] <= t_gap_end   + pd.Timedelta(minutes=30)].reset_index(drop=True)
adsc_gap  = adsc[(adsc["timestamp"] >= t_gap_start) &
                 (adsc["timestamp"] <= t_gap_end)].reset_index(drop=True)

print(f"\nADS-B before : {len(bef):>4} pts  (trimmed to {len(bef_trim)})")
print(f"ADS-C        : {len(adsc):>4} pts  ({len(adsc_gap)} inside gap)")
print(f"ADS-B after  : {len(aft):>4} pts  (trimmed to {len(aft_trim)})")
print(f"Gap          : {gap_min:.1f} min")
print(f"From         : lat={bef_trim['latitude'].iloc[-1]:.2f}  lon={bef_trim['longitude'].iloc[-1]:.2f}")
print(f"To           : lat={aft_trim['latitude'].iloc[0]:.2f}  lon={aft_trim['longitude'].iloc[0]:.2f}")

Flight : 20240312_346114_174632_193401
Step   : step1_raw_20240312_20240313

ADS-B before :  294 pts  (trimmed to 121)
ADS-C        :  431 pts  (431 inside gap)
ADS-B after  :   91 pts  (trimmed to 91)
Gap          : 390.0 min
From         : lat=37.37  lon=-13.98
To           : lat=18.61  lon=-64.62


## Cell 3 — Reconstruct the gap with all three methods

All three methods use only ADS-B before and after — ADS-C is never seen during reconstruction.

In [43]:
# ── Import new anchored Kalman from step5 module (always reload latest) ───────
import sys as _sys, importlib
_sys.path.insert(0, str(NOEL_DIR.parent / "notebook"))
import step5_kalman_aeroml3
importlib.reload(step5_kalman_aeroml3)
from step5_kalman_aeroml3 import reconstruct_single_kalman

# ── GRU reconstruction (same as step 5 evaluation) ────────────────────────────
def _normalize_seq(df):
    lat = (df["latitude"].values  - NORM["lat"]["mean"]) / NORM["lat"]["std"]
    lon = (df["longitude"].values - NORM["lon"]["mean"]) / NORM["lon"]["std"]
    vel = (df["velocity_mps"].values - NORM["vel"]["mean"]) / NORM["vel"]["std"]
    hdg = np.radians(df["heading_deg"].values)
    alt = (df["altitude"].values  - NORM["alt"]["mean"]) / NORM["alt"]["std"]
    return np.stack([lat, lon, vel, np.sin(hdg), np.cos(hdg), alt], axis=1).astype(np.float32)

def _resample_df(df, freq_sec):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = (df.dropna(subset=["latitude","longitude","altitude","velocity_mps","heading_deg"])
            .sort_values("timestamp").drop_duplicates("timestamp").set_index("timestamp"))
    cols = ["latitude","longitude","altitude","velocity_mps","heading_deg"]
    grid = pd.date_range(df.index[0], df.index[-1], freq=f"{freq_sec}s")
    if len(grid) < 2:
        return df[cols].reset_index().rename(columns={"index":"timestamp"})
    return df[cols].reindex(df[cols].index.union(grid)).interpolate("time").loc[grid].reset_index().rename(columns={"index":"timestamp"})

def _prepare_seq(df, n_steps, from_end):
    arr = _normalize_seq(df)
    arr = arr[-n_steps:] if from_end else arr[:n_steps]
    n_valid = arr.shape[0]
    pad = np.zeros((n_steps - n_valid, arr.shape[1]), dtype=np.float32)
    if from_end:
        arr  = np.vstack([pad, arr])
        mask = np.array([0.0]*(n_steps - n_valid) + [1.0]*n_valid, dtype=np.float32)
    else:
        arr  = np.vstack([arr, pad])
        mask = np.array([1.0]*n_valid + [0.0]*(n_steps - n_valid), dtype=np.float32)
    return arr, mask

def reconstruct_gap_gru(bef_df, aft_df, dt=15.0):
    bef_rs = _resample_df(bef_df, RESAMPLE_SEC)
    aft_rs = _resample_df(aft_df, RESAMPLE_SEC)
    if len(bef_rs) < 2 or len(aft_rs) < 2:
        return pd.DataFrame()

    bef_arr, bef_mask = _prepare_seq(bef_rs, BEFORE_STEPS, from_end=True)
    aft_arr, aft_mask = _prepare_seq(aft_rs, AFTER_STEPS,  from_end=False)

    t0 = bef_df["timestamp"].iloc[-1]; t1 = aft_df["timestamp"].iloc[0]
    gap_s = (t1 - t0).total_seconds()
    n_out = max(1, int(round(gap_s / dt)))
    taus  = np.array([(i+1)/(n_out+1) for i in range(n_out)], dtype=np.float32)

    la0 = np.array([float(bef_df["latitude"].iloc[-1])],  dtype=np.float32)
    lo0 = np.array([float(bef_df["longitude"].iloc[-1])], dtype=np.float32)
    la1 = np.array([float(aft_df["latitude"].iloc[0])],   dtype=np.float32)
    lo1 = np.array([float(aft_df["longitude"].iloc[0])],  dtype=np.float32)
    bl_lat, bl_lon = gc_interpolate_batch(la0, lo0, la1, lo1, taus[None, :])

    def _t(a): return torch.FloatTensor(a[None]).to(device)
    batch = {
        "before_seq":   _t(bef_arr), "before_mask": _t(bef_mask),
        "after_seq":    _t(aft_arr), "after_mask":  _t(aft_mask),
        "gap_norm":     torch.FloatTensor([gap_s/6000.]).to(device),
        "adsc_tau":     torch.FloatTensor(taus[None]).to(device),
        "baseline_lat": torch.FloatTensor(bl_lat.astype(np.float32)).to(device),
        "baseline_lon": torch.FloatTensor(bl_lon.astype(np.float32)).to(device),
    }
    with torch.no_grad():
        pred_lat, pred_lon = model_gru(batch)

    # Boundary alignment correction ------------------------------------
    # The GRU outputs points at tau in (0,1) exclusive.  Raw predictions
    # near the edges may not land on the ADS-B anchor coordinates, causing
    # visible jumps where the GRU segment meets the ADS-B track.
    # Fix: blend the residual error linearly so both endpoints land exactly
    # on the ADS-B before/after coordinates.  Internal shape is preserved.
    lat_np = pred_lat.cpu().numpy()[0].copy()   # (n_out,)
    lon_np = pred_lon.cpu().numpy()[0].copy()
    if n_out > 1:
        _t = np.linspace(0.0, 1.0, n_out)
        lat_np = lat_np + (1.0 - _t) * (float(bef_df["latitude"].iloc[-1])  - lat_np[0])  \
                        +        _t  * (float(aft_df["latitude"].iloc[0])    - lat_np[-1])
        lon_np = lon_np + (1.0 - _t) * (float(bef_df["longitude"].iloc[-1]) - lon_np[0])  \
                        +        _t  * (float(aft_df["longitude"].iloc[0])   - lon_np[-1])
    # n_out==1 edge case: single mid-gap point, leave where the model placed it

    alt0 = float(bef_df["altitude"].iloc[-1]); alt1 = float(aft_df["altitude"].iloc[0])
    return pd.DataFrame({
        "timestamp":  [t0 + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_out)],
        "latitude":   lat_np,
        "longitude":  lon_np,
        "altitude":   np.linspace(alt0, alt1, n_out),
        "interpolated": True,
    })

# ── Baseline: correct constant-speed great-circle interpolation ───────────────
def reconstruct_forward_only(before_df, after_df, dt=15.0):
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    n_steps = max(1, int(round((first_time - last_time).total_seconds() / dt)))
    lat0 = float(before_df["latitude"].iloc[-1]);  lon0 = float(before_df["longitude"].iloc[-1])
    alt0 = float(before_df["altitude"].iloc[-1])
    lat1 = float(after_df["latitude"].iloc[0]);    lon1 = float(after_df["longitude"].iloc[0])
    alt1 = float(after_df["altitude"].iloc[0])
    pts  = geod.npts(lon0, lat0, lon1, lat1, n_steps)
    lats = np.array([p[1] for p in pts])
    lons = np.array([p[0] for p in pts])
    alts = np.linspace(alt0, alt1, n_steps)
    timestamps = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    result = pd.DataFrame({"latitude": lats, "longitude": lons, "altitude": alts})
    result["timestamp"] = timestamps; result["interpolated"] = True
    return result

# ── Retime helper ─────────────────────────────────────────────────────────────
def retime_to_constant_speed(recon_df, before_df, after_df, dt=15.0):
    from scipy.interpolate import interp1d
    lats = recon_df["latitude"].values
    lons = recon_df["longitude"].values
    alts = recon_df["altitude"].values if "altitude" in recon_df.columns else np.zeros(len(recon_df))
    cum_dist = np.zeros(len(lats))
    for i in range(1, len(lats)):
        _, _, d = geod.inv(float(lons[i-1]), float(lats[i-1]), float(lons[i]), float(lats[i]))
        cum_dist[i] = cum_dist[i-1] + abs(d)
    total_dist = cum_dist[-1]
    if total_dist == 0:
        return recon_df
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    total_sec  = (first_time - last_time).total_seconds()
    n_steps    = max(1, int(round(total_sec / dt)))
    time_fracs   = np.array([dt*(i+1)/total_sec for i in range(n_steps)])
    target_fracs = np.clip(time_fracs, 0.0, 1.0)
    old_fracs    = cum_dist / total_dist
    f_la  = interp1d(old_fracs, lats, bounds_error=False, fill_value=(lats[0], lats[-1]))
    f_lo  = interp1d(old_fracs, lons, bounds_error=False, fill_value=(lons[0], lons[-1]))
    f_alt = interp1d(old_fracs, alts, bounds_error=False, fill_value=(alts[0], alts[-1]))
    new_ts = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    return pd.DataFrame({"latitude": f_la(target_fracs), "longitude": f_lo(target_fracs),
                         "altitude": f_alt(target_fracs), "timestamp": new_ts, "interpolated": True})

# ── Run all three methods ─────────────────────────────────────────────────────
print("Reconstructing gap with all three methods...")

recon_base = reconstruct_forward_only(bef_trim, aft_trim)

# Pass full bef (not bef_trim) so _get_entry_velocity can find the last real heading
recon_kalman_rt = reconstruct_single_kalman(bef, aft_trim, dt=15.0)

# Pass full bef so GRU gets its expected 64 min of before context.
# bef_trim was only 30 min (47% of training context), causing 34/64
# input slots to be zero-padded and degrading GRU predictions.
recon_gru = reconstruct_gap_gru(bef, aft_trim)

# ── Anchor verification ───────────────────────────────────────────────────────
from step5_kalman_aeroml3 import haversine_m as _hav
_d0 = _hav(recon_kalman_rt["latitude"].iloc[0],  recon_kalman_rt["longitude"].iloc[0],
           bef["latitude"].iloc[-1],              bef["longitude"].iloc[-1])
_d1 = _hav(recon_kalman_rt["latitude"].iloc[-1], recon_kalman_rt["longitude"].iloc[-1],
           aft_trim["latitude"].iloc[0],          aft_trim["longitude"].iloc[0])
from step5_kalman_aeroml3 import haversine_m as _hav2
_g0 = _hav2(recon_gru["latitude"].iloc[0],  recon_gru["longitude"].iloc[0],
            bef["latitude"].iloc[-1],         bef["longitude"].iloc[-1])
_g1 = _hav2(recon_gru["latitude"].iloc[-1], recon_gru["longitude"].iloc[-1],
            aft_trim["latitude"].iloc[0],     aft_trim["longitude"].iloc[0])
print(f"Baseline : {len(recon_base)} steps")
print(f"Kalman   : {len(recon_kalman_rt)} steps  start_gap={_d0:.1f}m  end_gap={_d1:.1f}m")
print(f"GRU      : {len(recon_gru)} steps  start_gap={_g0:.1f}m  end_gap={_g1:.1f}m")

Reconstructing gap with all three methods...
Baseline : 1560 steps
Kalman   : 1561 steps  start_gap=0.1m  end_gap=0.1m
GRU      : 1560 steps  start_gap=0.1m  end_gap=0.1m


## Cell 4 — Measure error against ADS-C ground truth

ADS-C was never used during reconstruction — it is now revealed to measure error.

In [44]:
def error_km(recon_df, truth_df):
    """
    Mean closest-point distance (km) between reconstruction and ADS-C truth.
    Uses nearest-neighbour matching so timestamp misalignment doesn't matter.
    """
    if len(truth_df) == 0 or len(recon_df) == 0:
        return float("nan")

    def haversine_m(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat, dlon = lat2-lat1, lon2-lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        return 2 * 6_371_000 * np.arcsin(np.sqrt(a))

    errs = []
    for _, row in truth_df.iterrows():
        dists = haversine_m(
            row["latitude"], row["longitude"],
            recon_df["latitude"].values, recon_df["longitude"].values,
        )
        errs.append(dists.min())
    return np.mean(errs) / 1000

base_err   = error_km(recon_base,       adsc_gap)
kalman_err = error_km(recon_kalman_rt,  adsc_gap)
gru_err    = error_km(recon_gru,        adsc_gap)

print(f"ADS-C waypoints used for evaluation: {len(adsc_gap)}")
print()
print(f"{'Method':<20} {'Mean error (km)':>16}")
print("-" * 38)
print(f"  {'Baseline':<18} {base_err:>14.1f} km")
print(f"  {'Kalman filter':<18} {kalman_err:>14.1f} km")
print(f"  {'GRU model':<18} {gru_err:>14.1f} km")
print()

best = min([("Baseline", base_err), ("Kalman", kalman_err), ("GRU", gru_err)],
           key=lambda x: x[1])
print(f"Best method: {best[0]} ({best[1]:.1f} km)")

ADS-C waypoints used for evaluation: 431

Method                Mean error (km)
--------------------------------------
  Baseline                    314.0 km
  Kalman filter               114.9 km
  GRU model                   167.5 km

Best method: Kalman (114.9 km)


## Cell 5 — Export GeoJSON for geojson.io

Exports all three methods + ADS-C ground truth as a single GeoJSON file.
Open at https://geojson.io

In [45]:
import json

def _line(df, lat_col, lon_col, props):
    coords = [[float(lo), float(la)] for la, lo in zip(df[lat_col], df[lon_col])
              if np.isfinite(float(la)) and np.isfinite(float(lo))]
    if len(coords) < 2:
        return None
    return {"type": "Feature", "properties": props,
            "geometry": {"type": "LineString", "coordinates": coords}}

features = []

# ADS-B boundary context (grey)
features.append(_line(bef_trim, "latitude", "longitude",
    {"label": "ADS-B before (context)",
     "stroke": "#888888", "stroke-width": 2, "stroke-opacity": 0.8}))
features.append(_line(aft_trim, "latitude", "longitude",
    {"label": "ADS-B after (context)",
     "stroke": "#888888", "stroke-width": 2, "stroke-opacity": 0.8}))

# ADS-C ground truth (yellow) — NOT used in reconstruction
features.append(_line(adsc_gap, "latitude", "longitude",
    {"label": f"ADS-C ground truth ({len(adsc_gap)} waypoints)",
     "stroke": "#FFC107", "stroke-width": 4, "stroke-opacity": 1.0}))

# Three independent reconstructions — none saw ADS-C
features.append(_line(recon_base, "latitude", "longitude",
    {"label": f"Baseline (great-circle) — {base_err:.1f} km",
     "stroke": "#F44336", "stroke-width": 2}))
features.append(_line(recon_kalman_rt, "latitude", "longitude",
    {"label": f"Kalman filter — {kalman_err:.1f} km",
     "stroke": "#00BCD4", "stroke-width": 3}))
features.append(_line(recon_gru, "latitude", "longitude",
    {"label": f"GRU model — {gru_err:.1f} km",
     "stroke": "#4CAF50", "stroke-width": 3}))

features = [f for f in features if f]
fc  = {"type": "FeatureCollection", "features": features}
out = OUT_DIR / f"{FLIGHT_NAME}_final_comparison.geojson"
out.write_text(json.dumps(fc, indent=2), encoding="utf-8")

print(f"GeoJSON saved → {out}")
print("Open at https://geojson.io")
print()
print("Legend:")
print("  Grey   = ADS-B track (boundary context, 30 min around gap)")
print("  Yellow = ADS-C ground truth (held-out, never used in reconstruction)")
print(f"  Red    = Baseline (great-circle)   — {base_err:.1f} km error")
print(f"  Cyan   = Kalman filter              — {kalman_err:.1f} km error")
print(f"  Green  = GRU model                  — {gru_err:.1f} km error")

GeoJSON saved → ..\outputs\20240312_346114_174632_193401_final_comparison.geojson
Open at https://geojson.io

Legend:
  Grey   = ADS-B track (boundary context, 30 min around gap)
  Yellow = ADS-C ground truth (held-out, never used in reconstruction)
  Red    = Baseline (great-circle)   — 314.0 km error
  Cyan   = Kalman filter              — 114.9 km error
  Green  = GRU model                  — 167.5 km error


## Cell 6 — (Optional) Find the best North Atlantic flight to visualize

In [46]:
import pandas as pd
from pathlib import Path

# -- Load full evaluation results (all flights, all three methods) ----------
# evaluation_results.csv is written by 05_evaluate.ipynb Cell 5 (the fixed
# version).  It covers ALL flights -- not just the test split -- because
# the fixed cell calls reconstruct_single_kalman() directly instead of
# looking up a split-limited pre-computed dict.
#
# GRU error here is the mean haversine nearest-point distance (km) from
# each ADS-C waypoint to the nearest GRU-reconstructed point -- identical
# metric to the Baseline and Kalman columns.
eval_path = Path('../outputs/evaluation_results.csv')
if not eval_path.exists():
    print('Run 05_evaluate.ipynb first to generate evaluation_results.csv.')
else:
    eval_df = pd.read_csv(eval_path)

    # Derived improvement columns
    eval_df['kalman_impr_km'] = eval_df['baseline_err_km'] - eval_df['kalman_err_km']
    eval_df['gru_impr_km']    = eval_df['baseline_err_km'] - eval_df['gru_err_km']
    eval_df['gru_impr_pct']   = (
        eval_df['gru_impr_km'] / eval_df['baseline_err_km'].clip(lower=1e-9) * 100
    )

    # -- Top 30 flights by GRU improvement (best visualisation candidates) --
    top = (
        eval_df.dropna(subset=['kalman_err_km', 'gru_err_km'])
        .sort_values('gru_impr_km', ascending=False)
        .head(30)
        [['flight', 'gap_minutes',
          'baseline_err_km', 'kalman_err_km', 'kalman_impr_km',
          'gru_err_km', 'gru_impr_km', 'gru_impr_pct',
          'adsc_waypoints']]
        .reset_index(drop=True)
    )

    print('Top 30 flights -- largest GRU improvement over baseline (km)')
    print('Columns: *_err_km = mean haversine nearest-point error vs ADS-C truth')
    print('         *_impr_km = baseline_err - method_err  (positive = better)')
    print('Copy a flight name from the "flight" column into Cell 2.\n')
    display(top)

    # -- Dataset-wide summary across all methods ---------------------------
    print('\n-- Summary across all flights --')
    base_mean = eval_df['baseline_err_km'].mean()
    for label, col in [('Baseline', 'baseline_err_km'),
                        ('Kalman',   'kalman_err_km'),
                        ('GRU',      'gru_err_km')]:
        vals = eval_df[col].dropna()
        imp = '' if label == 'Baseline' else f'  ({(1-vals.mean()/base_mean)*100:+.1f}% vs baseline)'
        print(f'  {label:<12}: mean={vals.mean():.1f}km  '
              f'median={vals.median():.1f}km  n={len(vals)}{imp}')

    # -- Flights where Kalman is worse than baseline (useful diagnostic) ----
    print('\n-- Flights where Kalman is WORSE than baseline (top 10) --')
    worse_k = (
        eval_df.dropna(subset=['kalman_err_km'])
        [eval_df['kalman_err_km'] > eval_df['baseline_err_km']]
        .sort_values('kalman_impr_km', ascending=True)
        .head(10)
        [['flight', 'gap_minutes',
          'baseline_err_km', 'kalman_err_km', 'gru_err_km',
          'kalman_impr_km']]
        .reset_index(drop=True)
    )
    display(worse_k)


Top 30 flights -- largest GRU improvement over baseline (km)
Columns: *_err_km = mean haversine nearest-point error vs ADS-C truth
         *_impr_km = baseline_err - method_err  (positive = better)
Copy a flight name from the "flight" column into Cell 2.



,flight,gap_minutes,baseline_err_km,kalman_err_km,kalman_impr_km,gru_err_km,gru_impr_km,gru_impr_pct,adsc_waypoints
0,20250801_400684_195403_220753,313.0,611.130206,200.971823,410.158383,281.967540,329.162666,53.861299,536
1,20241211_4005ba_190259_205554,307.2,359.672167,29.586131,330.086036,53.445002,306.227165,85.140634,453
2,20250703_4005c0_202353_224235,302.0,552.607515,160.536815,392.070700,260.873221,291.734295,52.792314,556
3,20250115_34740f_194451_205203,406.0,364.400763,136.374198,228.026565,77.619465,286.781297,78.699423,270
4,20250801_400774_203353_234805,316.2,568.816068,299.918631,268.897437,287.626511,281.189558,49.434180,778
5,20250801_4005bb_105232_131959,313.8,483.371580,89.983572,393.388008,205.212954,278.158626,57.545507,590
6,20250731_4b1885_115838_141058,323.8,383.015240,288.561834,94.453406,105.409504,277.605736,72.479031,530
7,20250731_aa92f9_115708_145110,330.5,460.173456,295.103163,165.070292,188.907416,271.266040,58.948650,697
8,20250801_4007f1_173814_202441,315.2,454.873635,334.623194,120.250441,183.776819,271.096815,59.598270,667
9,20250319_aa1a16_145849_164508,368.2,324.358362,148.165168,176.193194,55.282547,269.075815,82.956337,426



-- Summary across all flights --
  Baseline    : mean=119.9km  median=99.3km  n=2141
  Kalman      : mean=87.1km  median=67.0km  n=2141  (+27.3% vs baseline)
  GRU         : mean=86.6km  median=70.4km  n=2141  (+27.8% vs baseline)

-- Flights where Kalman is WORSE than baseline (top 10) --


,flight,gap_minutes,baseline_err_km,kalman_err_km,gru_err_km,kalman_impr_km
0,20250319_39cf11_144206_180022,411.2,29.782974,562.876289,11.416343,-533.093315
1,20230816_7380c1_065449_081655,251.8,86.400253,437.086676,108.291169,-350.686423
2,20240312_7103d6_155901_162237,309.5,13.646801,350.968026,57.763449,-337.321225
3,20250408_aabf80_131529_143041,269.5,127.696867,406.789325,93.436830,-279.092458
4,20230714_8964bc_145443_155730,421.0,83.780929,351.095907,63.041448,-267.314978
5,20240528_ab6ea5_125837_140242,276.0,25.216184,290.540461,63.081464,-265.324276
6,20230723_896488_142610_151647,407.2,57.485354,320.352321,25.288783,-262.866967
7,20250215_346114_182233_203701,390.2,401.695382,651.184008,195.502579,-249.488627
8,20231111_4bb184_034738_045702,259.5,79.768732,312.633345,102.942934,-232.864613
9,20230830_4ba9fa_084356_100539,269.0,42.053212,268.640199,20.760797,-226.586987


## Cell 7 — Test-set evaluation summary

Mean error of every flight on all three models vs ADS-C ground truth.

* **GRU** — tested on the 291 held-out test flights (never seen during training or validation)
* **Baseline & Kalman** — filtered to the same 291 flights for a fair apples-to-apples comparison

Requires `05_evaluate.ipynb` to have been run first (produces `outputs/evaluation_results.csv`).

In [47]:
import numpy as np
import pandas as pd
from pathlib import Path

# -- Identify the 291 held-out test flights --------------------------------
# sequences_test.npz was built by 04_step4_dataset.ipynb.
# GRU was trained on the train split only; test flights were never seen.
STEP4_DIR  = Path(r'C:\Users\marko\Desktop\AeroML3(og)\AeroML3\artifacts\step4_ml_dataset\dataset')
test_npz   = np.load(STEP4_DIR / 'sequences_test.npz', allow_pickle=True)
# segment_ids format: 'step_dir/flight_name'  -> keep only flight_name
test_names = {str(s).split('/')[-1] for s in test_npz['segment_ids']}
print(f'Test split : {len(test_names)} flights')

# -- Load full per-flight evaluation results --------------------------------
eval_path = Path('../outputs/evaluation_results.csv')
if not eval_path.exists():
    print('evaluation_results.csv not found — run 05_evaluate.ipynb first.')
else:
    eval_df = pd.read_csv(eval_path)

    # Filter to test-split flights so GRU comparison is fair
    test_df = eval_df[eval_df['flight'].isin(test_names)].copy()
    print(f'Matched     : {len(test_df)} flights in evaluation_results.csv')
    print(f'Total (all) : {len(eval_df)} flights\n')

    if test_df.empty:
        print('No test flights found — re-run 05_evaluate.ipynb.')
    else:
        # -- Aggregate summary table ---------------------------------------
        base_mean = test_df['baseline_err_km'].mean()
        print(f"{'Method':<20} {'Mean (km)':>10}  {'Median (km)':>12}  "
              f"{'P90 (km)':>10}  {'vs Baseline':>12}  {'n':>5}")
        print('─' * 76)
        for label, col in [
            ('Baseline',  'baseline_err_km'),
            ('Kalman',    'kalman_err_km'),
            ('GRU',       'gru_err_km'),
        ]:
            vals = test_df[col].dropna()
            imp  = '—' if label == 'Baseline' else f'{(1 - vals.mean()/base_mean)*100:+.1f}%'
            print(f'  {label:<18} {vals.mean():>10.1f}  {vals.median():>12.1f}  '
                  f'{vals.quantile(0.9):>10.1f}  {imp:>12}  {len(vals):>5}')

        k_sub  = test_df.dropna(subset=['kalman_err_km'])
        g_sub  = test_df.dropna(subset=['gru_err_km'])
        k_wins = (k_sub['kalman_err_km'] < k_sub['baseline_err_km']).sum()
        g_wins = (g_sub['gru_err_km']    < g_sub['baseline_err_km']).sum()
        print(f'\n  Kalman beats baseline on {k_wins}/{len(k_sub)} test flights '
              f'({k_wins/len(k_sub)*100:.1f}%)')
        print(f'  GRU    beats baseline on {g_wins}/{len(g_sub)} test flights '
              f'({g_wins/len(g_sub)*100:.1f}%)')

        # -- Per-flight breakdown table ------------------------------------
        show = test_df[['flight', 'gap_minutes', 'adsc_waypoints',
                         'baseline_err_km', 'kalman_err_km', 'gru_err_km']].copy()
        # Mark which method wins each flight
        show['best'] = show[['baseline_err_km', 'kalman_err_km', 'gru_err_km']] \
                           .idxmin(axis=1) \
                           .str.replace('_err_km', '', regex=False)
        print(f'\nPer-flight results — {len(show)} test flights:')
        display(show.sort_values('gap_minutes', ascending=False)
                    .round(1)
                    .reset_index(drop=True))


Test split : 289 flights
Matched     : 291 flights in evaluation_results.csv
Total (all) : 2141 flights

Method                Mean (km)   Median (km)    P90 (km)   vs Baseline      n
────────────────────────────────────────────────────────────────────────────
  Baseline                121.7         101.6       225.4             —    291
  Kalman                   88.0          67.6       182.1        +27.7%    291
  GRU                      93.9          75.0       167.2        +22.8%    291

  Kalman beats baseline on 203/291 test flights (69.8%)
  GRU    beats baseline on 231/291 test flights (79.4%)

Per-flight results — 291 test flights:


,flight,gap_minutes,adsc_waypoints,baseline_err_km,kalman_err_km,gru_err_km,best
0,20250115_484371_013533_031135,520.0,385,104.4,162.5,70.7,gru
1,20250319_76cd41_145854_185254,515.0,937,45.3,56.6,77.4,baseline
2,20250408_3965a2_133547_145420,435.0,315,20.1,22.3,126.9,baseline
3,20250318_06a0ad_214418_235137,429.0,510,426.6,130.1,430.4,kalman
4,20250214_4bb0e5_211201_013458,427.0,1052,646.2,366.9,656.8,kalman
...,...,...,...,...,...,...,...
286,20230903_76cce7_095600_110521,179.2,278,74.5,36.6,81.2,kalman
287,20240822_ab5c12_094121_104256,176.2,247,67.0,31.3,64.2,kalman
288,20240703_71010f_111510_124341,175.2,355,178.8,119.3,145.6,kalman
289,20240406_76cce7_171910_182523,172.5,266,53.3,39.4,23.0,gru
